In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

from FEX.models import fex
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from generate_data import make_data, make_adjacency
timesteps=5000
adj_matrix = make_adjacency(100, 3)
timeseries, t_derivs = make_data(num_samples=5000, adjacency=adj_matrix, snr=None, coupling=0.8)

In [ ]:
# print average degree
print(adj_matrix.sum() / adj_matrix.size(0))

In [ ]:
dimx_fex = fex.CoupledFEX('depth_2_tree_config', 'depth_2_tree_config', 0, controller_epochs=200, controller_lr=0.005, finetune_epochs=20000, finetune_lr=1e-4, num_fex_epochs=120, self_lr=0.02, inter_lr=0.02, bfgs_epochs=20, bfgs_lr=1.0, poolsize=8, device=device, expression_threshold=0.1)
dimx_fex.fit(timeseries, t_derivs, adj_matrix, num_workers=5, finetune_bs=512)

In [ ]:
dimy_fex = fex.SingleFEX('depth_3_leaves_4_config', 1, num_finetune_epochs=5000, controller_epochs=200, num_fex_epochs=80, device=device)
dimy_fex.fit(timeseries, t_derivs, num_workers=5)

In [ ]:
dimz_fex = fex.SingleFEX('depth_3_leaves_4_config', 2, num_finetune_epochs=10000, num_fex_epochs=100, finetune_lr=4e-4, device=device)
dimz_fex.fit(timeseries, t_derivs, num_workers=5)

In [ ]:
print(dimx_fex)
print(dimy_fex)
print(dimz_fex)

In [ ]:
predicted_states = torch.zeros(timesteps + 1, timeseries.size(1), timeseries.size(2), device=device)
predicted_states[0] = timeseries[0]
dt = 0.01
with torch.no_grad():
    for t in range(timesteps):
        state = predicted_states[t]

        dx_dt = torch.cat([
            dimx_fex.predict(state, adj_matrix.to(device)), # dimx_fex.predict(state, adj_matrix),
            dimy_fex.predict(state),
            dimz_fex.predict(state)
        ], dim=-1)

        predicted_states[t+1] = state + dt * dx_dt

        if not torch.isfinite(predicted_states[t+1]).all():
            print(f"Non-finite state at timestep {t + 1}")
            break
        
from FEX.utils.plots import plot_dynamics

node = 80
fig = plot_dynamics(timeseries[:, node, 0].cpu(), timeseries[:, node, 1].cpu(), timeseries[:, node, 2].cpu(), predicted_states[:, node, :].cpu(), elev=15, azim=75)
fig.show()


In [ ]:
import matplotlib.pyplot as plt

adj_matrix = make_adjacency(100, 3)

coupled_data, coupled_derivatives = make_data(
    num_samples=timesteps,
    adjacency=adj_matrix,
    coupling=0.8,
)

uncoupled_data, uncoupled_derivatives = make_data(
    num_samples=timesteps,
    adjacency=adj_matrix,
    coupling=0.0,
)

# axs has shape (2, 5)
fig, axs = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": "3d"})
num_nodes = 5
for node in range(num_nodes):
    axs[0].plot(
        coupled_data[:, node * 5, 0],
        coupled_data[:, node * 5, 1],
        coupled_data[:, node * 5, 2],
    )
axs[0].set_title(f"Coupled")
axs[0].view_init(elev=60, azim=120)
for node in range(num_nodes):
    axs[1].plot(
        uncoupled_data[:, node * 5, 0],
        uncoupled_data[:, node * 5, 1],
        uncoupled_data[:, node * 5, 2],
    )
axs[1].set_title(f"Uncoupled")
axs[1].view_init(elev=60, azim=120)

plt.tight_layout()
plt.show()

fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for node in range(5):
    axs[0].plot(uncoupled_data[:, node, 0], alpha=0.8)

for node in range(5):
    axs[1].plot(coupled_data[:, node, 0], alpha=0.8)

axs[0].set_title("Uncoupled oscillators")
axs[1].set_title("Coupled oscillators")

axs[0].set_ylabel("$x_i(t)$")
axs[1].set_ylabel("$x_i(t)$")
axs[1].set_xlabel("Timestep")

plt.tight_layout()
plt.show()